In [56]:
!pip install -U -q ultralytics


In [57]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [58]:
# Config & Setup
import os
import numpy as np
import pickle
from tqdm import tqdm

DATASET_PATH = "/content/drive/MyDrive/UPRIGHT_KINEMATIC"
SAVE_PATH = "/content/drive/MyDrive/Kinematic_Features"
os.makedirs(SAVE_PATH, exist_ok=True)

TARGET_CLASSES = [
    'tricep Pushdown', 'shoulder_stretch', 'shoulder press',
    'rope_skipping', 'Punching', 'pull Up', 'jumping_jacks'
]

OBJECT_CLASSES = [
    'none', 'cup', 'bottle', 'sports ball', 'cell phone',
    'fork', 'spoon', 'knife', 'bowl', 'sandwich',
    'remote', 'book', 'keyboard', 'laptop', 'scissors'
]

print("✅ Config loaded!")


✅ Config loaded!


In [59]:
# Load pre-extracted poses from .npy files
all_poses = []
all_labels = []
all_active_objects = []

print("Starting Feature Extraction from .npy files...\n")

for cls_idx, class_name in enumerate(TARGET_CLASSES):
    class_folder = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_folder):
        print(f"⏩ Skipping {class_name} - Folder not found")
        continue

    npy_files = [f for f in os.listdir(class_folder) if f.endswith('.npy')]
    print(f"🎥 Processing: {class_name} ({len(npy_files)} files)...")

    for npy_file in tqdm(npy_files):
        npy_path = os.path.join(class_folder, npy_file)
        pose_data = np.load(npy_path)  # Shape: (frames, 34)

        # Reshape from (frames, 34) to (frames, 17, 2)
        num_frames = pose_data.shape[0]
        poses = pose_data.reshape(num_frames, 17, 2)

        if num_frames > 5:
            all_poses.append(poses[:64])  # Cap at 64 frames
            all_labels.append(cls_idx)
            all_active_objects.append("none")

print(f"\n✅ Loaded {len(all_poses)} samples across {len(TARGET_CLASSES)} classes")
for idx, cls_name in enumerate(TARGET_CLASSES):
    count = sum(1 for l in all_labels if l == idx)
    print(f"  {cls_name:25s} → {count} samples")


Starting Feature Extraction from .npy files...

🎥 Processing: tricep Pushdown (62 files)...


100%|██████████| 62/62 [00:00<00:00, 284.86it/s]


🎥 Processing: shoulder_stretch (51 files)...


100%|██████████| 51/51 [00:00<00:00, 250.19it/s]


🎥 Processing: shoulder press (28 files)...


...

🎥 Processing: pull Up (126 files)...


100%|██████████| 126/126 [00:01<00:00, 94.97it/s] 


🎥 Processing: jumping_jacks (133 files)...


100%|██████████| 133/133 [00:01<00:00, 124.74it/s]


✅ Loaded 1001 samples across 7 classes
  tricep Pushdown           → 62 samples
  shoulder_stretch          → 51 samples
  shoulder press            → 28 samples
  rope_skipping             → 144 samples
  Punching                  → 457 samples
  pull Up                   → 126 samples
  jumping_jacks             → 133 samples


In [60]:
# 89-D Feature Engine + Split FIRST, then Augment ONLY training
from tensorflow.keras.utils import pad_sequences, to_categorical
from sklearn.model_selection import train_test_split
import random

OBJECT_CLASSES = [
    'none', 'cup', 'bottle', 'sports ball', 'cell phone',
    'fork', 'spoon', 'knife', 'bowl', 'sandwich',
    'remote', 'book', 'keyboard', 'laptop', 'scissors'
]

def build_89d_features(poses_list, objects_list):
    X = []
    for video_poses, video_obj in zip(poses_list, objects_list):
        obj_vector = np.zeros(15)
        if video_obj in OBJECT_CLASSES:
            obj_vector[OBJECT_CLASSES.index(video_obj)] = 1.0
        else:
            obj_vector[0] = 1.0

        vid_features = []
        prev_norm_kpts = None
        for i in range(len(video_poses)):
            current_kpts = video_poses[i]
            nose = current_kpts[0]
            l_wrist = current_kpts[9]; r_wrist = current_kpts[10]
            l_shoulder = current_kpts[5]; r_shoulder = current_kpts[6]

            shoulder_dist = np.linalg.norm(l_shoulder - r_shoulder)
            scale = shoulder_dist if shoulder_dist > 0.01 else 1.0

            norm_kpts = np.copy(current_kpts)
            if np.sum(current_kpts) > 0:
                norm_kpts = (norm_kpts - current_kpts[0]) / scale
            skeleton_flat = norm_kpts.flatten()

            if prev_norm_kpts is None:
                velocity_flat = np.zeros(34)
            else:
                velocity_flat = (norm_kpts - prev_norm_kpts).flatten()
            prev_norm_kpts = norm_kpts

            dist_l = np.linalg.norm(nose - l_wrist) / scale
            dist_r = np.linalg.norm(nose - r_wrist) / scale
            dir_l = (nose - l_wrist) / scale
            dir_r = (nose - r_wrist) / scale

            frame_matrix = np.concatenate([
                skeleton_flat, velocity_flat, obj_vector,
                [dist_l, dist_r], dir_l, dir_r
            ])
            vid_features.append(frame_matrix)
        X.append(vid_features)
    return X

def augment_pose(poses):
    augmented = []
    # Gaussian noise
    augmented.append(poses + np.random.normal(0, 0.005, poses.shape))
    # Scale variation
    augmented.append(poses * random.uniform(0.85, 1.15))
    # Horizontal flip
    flipped = poses.copy()
    flipped[:, :, 0] = 1.0 - flipped[:, :, 0]
    for l, r in [(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]:
        flipped[:, l], flipped[:, r] = flipped[:, r].copy(), flipped[:, l].copy()
    augmented.append(flipped)
    # Time stretch
    n = len(poses)
    new_len = int(n * random.uniform(0.8, 1.2))
    indices = np.linspace(0, n-1, new_len).astype(int)
    stretched = poses[indices][:64]
    if len(stretched) > 5:
        augmented.append(stretched)
    return augmented

# ===== STEP 1: Split ORIGINAL data first (no augmentation) =====
print("Step 1: Splitting original data...\n")

indices = list(range(len(all_poses)))
idx_temp, idx_test = train_test_split(indices, test_size=0.15, random_state=42, stratify=all_labels)
temp_labels = [all_labels[i] for i in idx_temp]
idx_train, idx_val = train_test_split(idx_temp, test_size=0.18, random_state=42, stratify=temp_labels)

train_poses = [all_poses[i] for i in idx_train]
train_labels = [all_labels[i] for i in idx_train]
train_objects = [all_active_objects[i] for i in idx_train]

val_poses = [all_poses[i] for i in idx_val]
val_labels = [all_labels[i] for i in idx_val]
val_objects = [all_active_objects[i] for i in idx_val]

test_poses = [all_poses[i] for i in idx_test]
test_labels = [all_labels[i] for i in idx_test]
test_objects = [all_active_objects[i] for i in idx_test]

print(f"Before augmentation: Train={len(train_poses)} Val={len(val_poses)} Test={len(test_poses)}")

# ===== STEP 2: Augment ONLY training data =====
print("\nStep 2: Augmenting training data only...\n")

class_counts = {}
for l in train_labels:
    class_counts[l] = class_counts.get(l, 0) + 1

max_count = max(class_counts.values())

aug_poses = []
aug_labels = []
aug_objects = []

for cls_idx in range(len(TARGET_CLASSES)):
    cls_data = [(p, o) for p, l, o in zip(train_poses, train_labels, train_objects) if l == cls_idx]
    needed = max_count - len(cls_data)
    if needed > 0:
        print(f"📈 {TARGET_CLASSES[cls_idx]:25s}: {len(cls_data)} → +{needed} augmented")
        added = 0
        while added < needed:
            orig_pose, orig_obj = random.choice(cls_data)
            for aug in augment_pose(orig_pose):
                aug_poses.append(aug)
                aug_labels.append(cls_idx)
                aug_objects.append(orig_obj)
                added += 1
                if added >= needed: break

train_poses += aug_poses
train_labels += aug_labels
train_objects += aug_objects

print(f"\nAfter augmentation: Train={len(train_poses)} Val={len(val_poses)} Test={len(test_poses)}")

# ===== STEP 3: Build 89-D features =====
print("\n⚙️ Building 89-D features...")

X_train_raw = build_89d_features(train_poses, train_objects)
X_val_raw = build_89d_features(val_poses, val_objects)
X_test_raw = build_89d_features(test_poses, test_objects)

X_train_m = pad_sequences(X_train_raw, maxlen=64, dtype='float32', padding='post', value=0.0)
X_val_m = pad_sequences(X_val_raw, maxlen=64, dtype='float32', padding='post', value=0.0)
X_test_m = pad_sequences(X_test_raw, maxlen=64, dtype='float32', padding='post', value=0.0)

y_train_m = to_categorical(train_labels, num_classes=len(TARGET_CLASSES))
y_val_m = to_categorical(val_labels, num_classes=len(TARGET_CLASSES))
y_test_m = to_categorical(test_labels, num_classes=len(TARGET_CLASSES))

print(f"\n✅ X_train: {X_train_m.shape} | X_val: {X_val_m.shape} | X_test: {X_test_m.shape}")


Step 1: Splitting original data...

Before augmentation: Train=697 Val=153 Test=151

Step 2: Augmenting training data only...

📈 tricep Pushdown          : 43 → +275 augmented
📈 shoulder_stretch         : 35 → +283 augmented
📈 shoulder press           : 20 → +298 augmented
📈 rope_skipping            : 100 → +218 augmented
📈 pull Up                  : 88 → +230 augmented
📈 jumping_jacks            : 93 → +225 augmented

After augmentation: Train=2226 Val=153 Test=151

⚙️ Building 89-D features...

✅ X_train: (2226, 64, 89) | X_val: (153, 64, 89) | X_test: (151, 64, 89)


In [61]:
# Train BiGRU Model — Anti-Overfitting + Class Balanced
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to handle imbalance
train_labels = np.argmax(y_train_m, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = dict(enumerate(class_weights))
print("📊 Class weights:")
for k, v in class_weight_dict.items():
    print(f"  {TARGET_CLASSES[k]:25s} → {v:.2f}x")

model_89 = models.Sequential([
    layers.Input(shape=(64, 89)),
    layers.Masking(mask_value=0.0),
    layers.GaussianNoise(0.03),
    layers.Bidirectional(layers.GRU(48, return_sequences=True)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Bidirectional(layers.GRU(24)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(24, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.02)),
    layers.Dropout(0.3),
    layers.Dense(len(TARGET_CLASSES), activation='softmax')
])

model_89.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

print("\n🚀 Training Class-Balanced Model...\n")
history = model_89.fit(
    X_train_m, y_train_m,
    epochs=60,
    batch_size=16,
    validation_data=(X_val_m, y_val_m),
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict
)

print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]*100:.2f}%")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")


📊 Class weights:
  tricep Pushdown           → 1.00x
  shoulder_stretch          → 1.00x
  shoulder press            → 1.00x
  rope_skipping             → 1.00x
  Punching                  → 1.00x
  pull Up                   → 1.00x
  jumping_jacks             → 1.00x

🚀 Training Class-Balanced Model...

Epoch 1/60
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.3378 - loss: 2.4104 - val_accuracy: 0.4902 - val_loss: 1.9629 - learning_rate: 0.0010
Epoch 2/60
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.5431 - loss: 1.7088 - val_accuracy: 0.6797 - val_loss: 1.4401 - learning_rate: 0.0010
Epoch 3/60
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.6280 - loss: 1.4112 - val_accuracy: 0.7843 - val_loss: 1.1726 - learning_rate: 0.0010
Epoch 4/60
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7224 - loss: 1.1623 - val_accuracy: 0.8562 - val_loss: 0.8309 - learning_rate: 0.0010
Epoch 5/60
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7871 - loss: 0.

In [62]:
# Print Final Accuracy
print(f"Final Training Accuracy: {history.history['accuracy'][-1]*100:.2f}%")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")


Final Training Accuracy: 97.08%
Final Validation Accuracy: 90.85%


In [63]:
# Per-Class Evaluation
base_probs = model_89.predict(X_test_m, verbose=0)
base_preds = np.argmax(base_probs, axis=1)
y_true = np.argmax(y_test_m, axis=1)

base_acc = np.mean(base_preds == y_true)
print(f"📊 89-Feature Model Accuracy: {base_acc*100:.2f}%")

print("\n📋 Per-Class Breakdown:")
for idx, cls_name in enumerate(TARGET_CLASSES):
    mask = (y_true == idx)
    if mask.sum() > 0:
        cls_acc = np.mean(base_preds[mask] == y_true[mask])
        print(f"  {cls_name:25s} → {cls_acc*100:.1f}% ({mask.sum()} samples)")


📊 89-Feature Model Accuracy: 88.74%

📋 Per-Class Breakdown:
  tricep Pushdown           → 77.8% (9 samples)
  shoulder_stretch          → 62.5% (8 samples)
  shoulder press            → 75.0% (4 samples)
  rope_skipping             → 86.4% (22 samples)
  Punching                  → 94.2% (69 samples)
  pull Up                   → 94.7% (19 samples)
  jumping_jacks             → 85.0% (20 samples)


In [64]:
# Test on a single .npy sample
TEST_FOLDER = "/content/drive/MyDrive/UPRIGHT_KINEMATIC/jumping_jacks/"
test_files = [f for f in os.listdir(TEST_FOLDER) if f.endswith('.npy')]
test_path = os.path.join(TEST_FOLDER, test_files[0])

test_pose_data = np.load(test_path).reshape(-1, 17, 2)[:64]

test_vid_features = []
prev_norm_kpts_test = None
obj_vector_test = np.zeros(15)
obj_vector_test[0] = 1.0

for i in range(len(test_pose_data)):
    current_kpts = test_pose_data[i]
    nose = current_kpts[0]
    l_wrist = current_kpts[9]; r_wrist = current_kpts[10]
    l_shoulder = current_kpts[5]; r_shoulder = current_kpts[6]

    shoulder_dist = np.linalg.norm(l_shoulder - r_shoulder)
    scale = shoulder_dist if shoulder_dist > 0.01 else 1.0

    norm_kpts = np.copy(current_kpts)
    if np.sum(current_kpts) > 0:
        norm_kpts = (norm_kpts - current_kpts[0]) / scale
    skeleton_flat = norm_kpts.flatten()

    if prev_norm_kpts_test is None:
        velocity_flat = np.zeros(34)
    else:
        velocity_flat = (norm_kpts - prev_norm_kpts_test).flatten()
    prev_norm_kpts_test = norm_kpts

    dist_l = np.linalg.norm(nose - l_wrist) / scale
    dist_r = np.linalg.norm(nose - r_wrist) / scale
    dir_l = (nose - l_wrist) / scale
    dir_r = (nose - r_wrist) / scale

    frame_matrix = np.concatenate([
        skeleton_flat, velocity_flat, obj_vector_test,
        [dist_l, dist_r], dir_l, dir_r
    ])
    test_vid_features.append(frame_matrix)

X_test_single = pad_sequences([test_vid_features], maxlen=64, dtype='float32', padding='post', value=0.0)

prediction_probs = model_89.predict(X_test_single, verbose=0)[0]
predicted_idx = np.argmax(prediction_probs)

print(f"File: {test_files[0]}")
print(f"Predicted: {TARGET_CLASSES[predicted_idx]} ({prediction_probs[predicted_idx]*100:.2f}%)")
print("\nAll probabilities:")
for i, prob in enumerate(prediction_probs):
    print(f"  {TARGET_CLASSES[i]:25s} → {prob*100:.2f}%")


File: Copy of 6689952-uhd_2160_3840_24fps_pose.npy
Predicted: Punching (64.90%)

All probabilities:
  tricep Pushdown           → 3.48%
  shoulder_stretch          → 0.26%
  shoulder press            → 0.09%
  rope_skipping             → 28.65%
  Punching                  → 64.90%
  pull Up                   → 0.41%
  jumping_jacks             → 2.21%


In [65]:
# Cell 11: Save Model to Google Drive
model_save_path = os.path.join(SAVE_PATH, 'kinematic_action_model.keras')
model_89.save(model_save_path)
print(f"✅ Model successfully saved to your Drive at: {model_save_path}")


✅ Model successfully saved to your Drive at: /content/drive/MyDrive/Kinematic_Features/kinematic_action_model.keras


In [66]:
# Cell 12: Download Model to your Local PC
from google.colab import files
try:
    files.download(model_save_path)
    print("⬇️ Downloading model to your computer...")
except Exception as e:
    print(f"❌ Download failed: {e}")
    print("You can manually download it by going to the 'Files' tab on the left in Colab, navigating to 'drive/MyDrive/Kinematic_Features/', right-clicking 'kinematic_action_model.keras', and selecting 'Download'.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Downloading model to your computer...


In [72]:
import cv2
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.utils import pad_sequences

# 1. Load YOLO Pose (it will download automatically if not already loaded)
print("Loading YOLOv8-pose...")
pose_model = YOLO('yolov8m-pose.pt')

# 2. Set your raw video path here (upload an .mp4 to colab first)
VIDEO_PATH = "/content/drive/MyDrive/v_JumpingJack_g25_c07.avi"
cap = cv2.VideoCapture(VIDEO_PATH)
raw_poses = []

print(f"Extracting skeletons from {VIDEO_PATH}...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Resize for consistent inference
    frame_resized = cv2.resize(frame, (1280, 720))
    results = pose_model(frame_resized, verbose=False)[0]

    # Extract 17 keypoints
    if results.keypoints is not None and len(results.keypoints.data) > 0:
        kpts = results.keypoints.data.cpu().numpy()[0][:, :2]
    else:
        kpts = np.zeros((17, 2))

    raw_poses.append(kpts)

    if len(raw_poses) >= 64: break # Cap at 64 frames (same as training)
cap.release()

if len(raw_poses) < 5:
    print("❌ Not enough frames or no person detected!")
else:
    print(f"✅ Extracted {len(raw_poses)} frames. Building 89-D features...")

    # 3. Build 89-D features
    test_features = []
    prev_norm_kpts = None
    obj_vector = np.zeros(15)
    obj_vector[0] = 1.0 # Set object to "none"

    for i in range(len(raw_poses)):
        current_kpts = raw_poses[i]
        nose = current_kpts[0]
        l_wrist = current_kpts[9]; r_wrist = current_kpts[10]
        l_shoulder = current_kpts[5]; r_shoulder = current_kpts[6]

        shoulder_dist = np.linalg.norm(l_shoulder - r_shoulder)
        scale = shoulder_dist if shoulder_dist > 0.01 else 1.0

        norm_kpts = np.copy(current_kpts)
        if np.sum(current_kpts) > 0:
            norm_kpts = (norm_kpts - current_kpts[0]) / scale
        skeleton_flat = norm_kpts.flatten()

        if prev_norm_kpts is None:
            velocity_flat = np.zeros(34)
        else:
            velocity_flat = (norm_kpts - prev_norm_kpts).flatten()
        prev_norm_kpts = norm_kpts

        dist_l = np.linalg.norm(nose - l_wrist) / scale
        dist_r = np.linalg.norm(nose - r_wrist) / scale
        dir_l = (nose - l_wrist) / scale
        dir_r = (nose - r_wrist) / scale

        # Combine into 89-D vector
        frame_matrix = np.concatenate([
            skeleton_flat, velocity_flat, obj_vector,
            [dist_l, dist_r], dir_l, dir_r
        ])
        test_features.append(frame_matrix)

    # 4. Predict using your trained model_89
    X_test_live = pad_sequences([test_features], maxlen=64, dtype='float32', padding='post', value=0.0)
    prediction_probs = model_89.predict(X_test_live, verbose=0)[0]
    predicted_idx = np.argmax(prediction_probs)

    print(f"\n🎥 Prediction Results:")
    print(f"🥇 Predicted Activity: {TARGET_CLASSES[predicted_idx]} ({prediction_probs[predicted_idx]*100:.2f}%)")

    print("\n📊 All class probabilities:")
    for i, prob in enumerate(prediction_probs):
        print(f"  {TARGET_CLASSES[i]:25s} → {prob*100:.2f}%")


Loading YOLOv8-pose...
Extracting skeletons from /content/drive/MyDrive/v_JumpingJack_g25_c07.avi...
✅ Extracted 64 frames. Building 89-D features...

🎥 Prediction Results:
🥇 Predicted Activity: jumping_jacks (95.41%)

📊 All class probabilities:
  tricep Pushdown           → 0.01%
  shoulder_stretch          → 0.07%
  shoulder press            → 0.08%
  rope_skipping             → 4.25%
  Punching                  → 0.09%
  pull Up                   → 0.10%
  jumping_jacks             → 95.41%
